# QDSV/QIntent Controlled qLDPC-Style Benchmark

This notebook is a controlled benchmark for the post-decoding role of QDSV/QIntent.

It builds a small sparse CSS/qLDPC-style syndrome map where a correlated two-qubit correction and a risky single-qubit correction produce the same syndrome. A minimum-weight / likelihood baseline prefers the singleton. QDSV structured semantic scoring re-ranks the same decoder-generated candidates using syndrome evidence, logical safety, decoder confidence, and risk.

This is still a controlled toy benchmark, not a hardware-scale qLDPC benchmark. The purpose is to validate the experimental pattern before connecting a real qLDPC decoder.

## 1. Install QIntent

In [ ]:
!pip install -q --upgrade "git+https://github.com/qdsvquantum-afk/qintent.git"

In [ ]:
import json
import math
from itertools import combinations
from pathlib import Path

import pandas as pd

from qintent import QIntentClient

API_URL = "https://qdsv-api-568788785178.us-central1.run.app/api"
client = QIntentClient(api_url=API_URL, timeout=60)

client.spec()["public_limits"]

## 2. Construct a sparse benchmark code

For each block, two safe qubits and one logical-sensitive qubit share a syndrome relation:

`H[sensitive] = H[safe_a] xor H[safe_b]`

This means the same observed syndrome can be explained by either:

- a high-confidence single-qubit correction on a logical-sensitive qubit; or
- a lower-confidence correlated two-qubit correction on safer qubits.

The simulated true error is the correlated two-qubit event. This lets us evaluate exact correction and logical-failure proxy, not only risk ranking.

In [ ]:
M_CHECKS = 14
N_BLOCKS = 6
MAX_CANDIDATE_WEIGHT = 3
PHYSICAL_ERROR_RATE = 0.07


def xor_vec(*vectors):
    out = [0] * M_CHECKS
    for vector in vectors:
        out = [a ^ b for a, b in zip(out, vector)]
    return tuple(out)


check_columns = {}
block_pairs = []
logical_sensitive_qubits = set()
logical_failure_sets = []

for block in range(N_BLOCKS):
    safe_a = 3 * block
    safe_b = safe_a + 1
    sensitive = safe_a + 2

    col_a = [0] * M_CHECKS
    col_b = [0] * M_CHECKS
    col_a[2 * block] = 1
    col_a[(2 * block + 6) % M_CHECKS] = 1
    col_b[2 * block + 1] = 1
    col_b[(2 * block + 7) % M_CHECKS] = 1

    check_columns[safe_a] = tuple(col_a)
    check_columns[safe_b] = tuple(col_b)
    check_columns[sensitive] = xor_vec(tuple(col_a), tuple(col_b))

    block_pairs.append((safe_a, safe_b, sensitive))
    logical_sensitive_qubits.add(sensitive)
    logical_failure_sets.append(frozenset((safe_a, safe_b, sensitive)))


pd.DataFrame([
    {
        "qubit": q,
        "check_column": "".join(str(bit) for bit in column),
        "logical_sensitive": q in logical_sensitive_qubits,
    }
    for q, column in check_columns.items()
])

## 3. Candidate generation and outcome labels

In [ ]:
def syndrome(qubits):
    return xor_vec(*(check_columns[q] for q in qubits))


def residual_error(true_error, correction):
    return frozenset(set(true_error) ^ set(correction))


def logical_failure_proxy(residual):
    return any(logical_op.issubset(residual) for logical_op in logical_failure_sets)


def generate_candidates(scenario_id, true_error):
    observed = syndrome(true_error)
    raw = []
    for weight in range(1, MAX_CANDIDATE_WEIGHT + 1):
        for qubits in combinations(check_columns.keys(), weight):
            if syndrome(qubits) != observed:
                continue
            overlap = sum(q in logical_sensitive_qubits for q in qubits)
            log_likelihood = weight * math.log(PHYSICAL_ERROR_RATE) + (len(check_columns) - weight) * math.log(1 - PHYSICAL_ERROR_RATE)
            residual = residual_error(true_error, qubits)
            raw.append((qubits, weight, overlap, log_likelihood, len(residual) == 0, logical_failure_proxy(residual)))

    likelihoods = [item[3] for item in raw]
    low, high = min(likelihoods), max(likelihoods)

    rows = []
    for candidate_index, (qubits, weight, overlap, log_likelihood, exact, logical_fail) in enumerate(raw):
        decoder_confidence = 1000 if high == low else int(round(600 + 400 * (log_likelihood - low) / (high - low)))
        rows.append({
            "scenario_id": scenario_id,
            "candidate_index": candidate_index,
            "correction_qubits": " ".join(str(q) for q in qubits),
            "candidate_weight": weight,
            "observed_syndrome": "".join(str(bit) for bit in observed),
            "predicted_syndrome": "".join(str(bit) for bit in observed),
            "syndrome_support": max(0, 1000 - 18 * max(0, weight - 1)),
            "check_consistency": 1000,
            "logical_preservation": max(0, 965 - 255 * overlap - 22 * max(0, weight - 1)),
            "distance_safety": max(0, 945 - 215 * overlap - 18 * weight),
            "decoder_confidence": decoder_confidence,
            "propagation_safety": max(0, 945 - 55 * weight - 65 * overlap),
            "syndrome_risk": min(1000, 30 + 10 * weight),
            "logical_risk": min(1000, 20 + 225 * overlap + 18 * weight),
            "syndrome_entropy_adjustment": 18 if weight == 2 and overlap == 0 else (-18 if overlap else 0),
            "exact_correction": exact,
            "logical_failure_proxy": logical_fail,
        })
    return rows


scenario_rows = []
scenario_metadata = []
for scenario_id, (safe_a, safe_b, sensitive) in enumerate(block_pairs):
    true_error = (safe_a, safe_b)
    rows = generate_candidates(scenario_id, true_error)
    scenario_rows.append(rows)
    scenario_metadata.append({
        "scenario_id": scenario_id,
        "true_error_qubits": " ".join(str(q) for q in true_error),
        "risky_singleton": str(sensitive),
        "observed_syndrome": rows[0]["observed_syndrome"],
        "candidate_count": len(rows),
    })

pd.DataFrame(scenario_metadata)

## 4. QIntent structured semantic score

In [ ]:
source = """
find_rows("candidate_index")
  .using_structured_semantic_score([
      block("syndrome", [
          signal("syndrome_support", influence=30, priority=2),
          signal("check_consistency", influence=20, priority=1),
      ], influence=30, priority=2, risk_adjustment="syndrome_risk", adjustments=[
          adjustment("syndrome_entropy_adjustment", influence=5),
      ]),
      block("logical_safety", [
          signal("logical_preservation", influence=40, priority=3),
          signal("distance_safety", influence=20, priority=2),
      ], influence=40, priority=3, risk_adjustment="logical_risk"),
      block("decoder", [
          signal("decoder_confidence", influence=25, priority=1),
          signal("propagation_safety", influence=15, priority=2),
      ], influence=30, priority=1),
  ], global_risk="logical_risk", profile="qldpc_formal_benchmark")
  .accept_if(threshold=600)
  .rank()
  .top_k(1)
"""

passport = client.explain(source, rows=scenario_rows[0])
predicate_summary = passport["semantic_execution_passport"]["predicate"]
assert predicate_summary["kind"] == "structured_semantic_score"
assert predicate_summary["blocks_count"] == 3
assert predicate_summary["signals_count"] == 6
assert predicate_summary["internal_formula_exposed"] is False
predicate_summary

## 5. Run benchmark

In [ ]:
summary_rows = []
selected_by_scenario = []

for meta, rows in zip(scenario_metadata, scenario_rows):
    candidates = pd.DataFrame(rows)
    baseline = candidates.sort_values(["decoder_confidence", "candidate_weight"], ascending=[False, True]).iloc[0]
    result = client.run(source, rows=rows)
    selected = result["result"]["selected_rows"][0]
    selected_by_scenario.append(selected)

    summary_rows.append({
        "scenario_id": meta["scenario_id"],
        "true_error_qubits": meta["true_error_qubits"],
        "risky_singleton": meta["risky_singleton"],
        "observed_syndrome": meta["observed_syndrome"],
        "candidate_count": meta["candidate_count"],
        "baseline_qubits": baseline["correction_qubits"],
        "baseline_confidence": int(baseline["decoder_confidence"]),
        "baseline_logical_risk": int(baseline["logical_risk"]),
        "baseline_exact": bool(baseline["exact_correction"]),
        "baseline_logical_failure_proxy": bool(baseline["logical_failure_proxy"]),
        "qdsv_qubits": selected["correction_qubits"],
        "qdsv_confidence": int(selected["decoder_confidence"]),
        "qdsv_logical_risk": int(selected["logical_risk"]),
        "qdsv_exact": bool(selected["exact_correction"]),
        "qdsv_logical_failure_proxy": bool(selected["logical_failure_proxy"]),
        "risk_reduction": int(baseline["logical_risk"]) - int(selected["logical_risk"]),
    })

summary = pd.DataFrame(summary_rows)
summary

## 6. Aggregate metrics

In [ ]:
metrics = {
    "scenarios": int(len(summary)),
    "baseline_exact_rate": float(summary["baseline_exact"].mean()),
    "qdsv_exact_rate": float(summary["qdsv_exact"].mean()),
    "baseline_logical_failure_proxy_rate": float(summary["baseline_logical_failure_proxy"].mean()),
    "qdsv_logical_failure_proxy_rate": float(summary["qdsv_logical_failure_proxy"].mean()),
    "baseline_avg_logical_risk": float(summary["baseline_logical_risk"].mean()),
    "qdsv_avg_logical_risk": float(summary["qdsv_logical_risk"].mean()),
    "avg_risk_reduction": float(summary["risk_reduction"].mean()),
}

metrics

## 7. Inspect one scenario

In [ ]:
scenario_id = 0
pd.DataFrame(scenario_rows[scenario_id]).sort_values(["decoder_confidence", "candidate_weight"], ascending=[False, True])

In [ ]:
print(json.dumps(selected_by_scenario[scenario_id]["_qdsv_decision_model"], indent=2))

## 8. Save evidence

In [ ]:
evidence = {
    "api_url": API_URL,
    "profile": "qldpc_formal_benchmark",
    "benchmark_design": {
        "m_checks": M_CHECKS,
        "n_blocks": N_BLOCKS,
        "max_candidate_weight": MAX_CANDIDATE_WEIGHT,
        "physical_error_rate": PHYSICAL_ERROR_RATE,
        "construction": "Each block has H[sensitive] = H[safe_a] xor H[safe_b]. True errors are correlated safe pairs.",
    },
    "check_columns": {str(k): list(v) for k, v in check_columns.items()},
    "logical_sensitive_qubits": sorted(logical_sensitive_qubits),
    "scenario_metadata": scenario_metadata,
    "qintent_source": source,
    "predicate_summary": predicate_summary,
    "summary": summary.to_dict(orient="records"),
    "metrics": metrics,
    "selected_by_scenario": selected_by_scenario,
    "internal_formula_exposed": False,
}

Path("qdsv_qldpc_formal_benchmark_evidence.json").write_text(json.dumps(evidence, indent=2), encoding="utf-8")
summary.to_csv("qdsv_qldpc_formal_benchmark_summary.csv", index=False)

print("Saved:")
print("- qdsv_qldpc_formal_benchmark_evidence.json")
print("- qdsv_qldpc_formal_benchmark_summary.csv")

## Interpretation

This controlled benchmark tests QDSV/QIntent as a risk-aware post-decoding decision layer. The baseline is intentionally likelihood/minimum-weight oriented. QDSV is given the same candidate set but can select the correlated correction when its structured evidence indicates lower logical risk and stronger logical safety.

This does not yet prove performance on production qLDPC codes. It does provide a reproducible bridge from toy demonstration to a more formal decoder-in-the-loop benchmark.